# ercot-bess-lab
### A battery dispatch backtester for the ERCOT power market

Grid-scale batteries make money two ways: buying cheap power and selling it back
expensive (**energy arbitrage**), and getting paid to hold reserve power in case the
grid operator needs it (**ancillary services**). The hard part isn't dispatching a
battery — it's knowing whether a given dispatch *strategy* is actually good, versus
just lucky or leaving money on the table.

This project builds that measuring stick the way a real trading desk would: pull
actual ERCOT market data, compute the **perfect-foresight revenue ceiling** (what a
battery could have earned with full hindsight), and eventually score every causal
(no-lookahead) strategy as **% of that ceiling captured**.

Everything below runs live against real ingested ERCOT data and the actual project
code — no numbers are hand-typed or mocked. Two milestones are complete so far:

| # | Milestone | What it proves |
|---|---|---|
| M1 | Ingestion + data quality | A month of real DAM/RTM/AS price and load data, validated clean |
| M2 | Perfect-foresight benchmark | An LP that finds the revenue-maximizing dispatch, given hindsight |

Planned next: causal strategies scored against this ceiling (M3), a longer history
spanning ERCOT's real-time market redesign (M4), and a polished report (M5). Full
detail lives in [`docs/milestones/`](../docs/milestones/) and
[`docs/adr/`](../docs/adr/) (architecture decision records — what was decided, why,
and what it costs).

## Architecture

```
ingest (gridstatus) -> silver (Polars, typed/cleaned) -> gold (analysis marts, DuckDB)
                                                              |
                                        +-----------------------------------+
                                        v                                   v
                         optimize: perfect foresight LP              optimize + forecast: causal strategies
                         (the revenue ceiling, built)                (DA-committed, rolling-horizon MPC)
                                        |                                   |
                                        +-----------------+-----------------+
                                                          v
                                              backtest: walk-forward engine,
                                              settlement math, % of perfect revenue
                                                          |
                                                          v
                                                report: HTML report per run
```

Real market data flows through a **medallion pipeline** (raw parquet cache -> typed,
DQ-validated silver tables -> DuckDB gold views), then into a CVXPY/HiGHS linear
program that solves for optimal battery dispatch.

In [1]:
import datetime as dt
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"

# resolve to repo root regardless of whether this runs from notebooks/ (Jupyter Lab
# default) or the repo root (nbconvert default)
if not (Path.cwd() / "data" / "ercot.duckdb").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, "src")

from ercot_bess.models.battery import BatterySpec
from ercot_bess.optimize.data_loading import load_dam_as_price_series, load_spp_series
from ercot_bess.optimize.perfect_foresight import solve_perfect_foresight
from ercot_bess.transform.dq import run_dq_checks

con = duckdb.connect("data/ercot.duckdb", read_only=True)
con.execute("PRAGMA disable_progress_bar")

HUB = "HB_HOUSTON"
START, END = dt.date(2025, 6, 1), dt.date(2025, 6, 30)
battery = BatterySpec()  # 100MW / 200MWh, 86% round-trip efficiency, $2/MWh degradation
battery

BatterySpec(power_mw=100.0, energy_mwh=200.0, round_trip_efficiency=0.86, min_soc_fraction=0.0, max_soc_fraction=1.0, daily_cycle_limit=1.5, degradation_cost_per_mwh=2.0)

## 1. Real market data, validated

One month of ERCOT data for **HB_HOUSTON** — a coastal trading hub with real
congestion-driven volatility, and a recognizable industry benchmark (real BESS fleets
concentrate here and at HB_WEST; see
[ADR 0005](../docs/adr/0005-default-settlement-point.md)). June 2025 is deliberately
*pre*-RTC+B, ERCOT's real-time co-optimization redesign that goes live
2025-12-05 — comparing before/after is M4.

Day-ahead (DAM) is the hourly-committed price; real-time (RTM) is the 15-minute price
actual dispatch settles against. The gap between them, and RTM's extra volatility, is
the raw material every arbitrage strategy in this project trades on.

In [2]:
dam = con.execute(
    "SELECT interval_start, spp_usd_per_mwh FROM silver_dam_spp WHERE location = ? ORDER BY interval_start",
    [HUB],
).df()
rtm = con.execute(
    "SELECT interval_start, spp_usd_per_mwh FROM silver_rtm_spp WHERE location = ? ORDER BY interval_start",
    [HUB],
).df()

fig = go.Figure()
fig.add_trace(go.Scatter(x=rtm["interval_start"], y=rtm["spp_usd_per_mwh"],
                          name="RTM (15-min)", line=dict(color="#636EFA", width=1)))
fig.add_trace(go.Scatter(x=dam["interval_start"], y=dam["spp_usd_per_mwh"],
                          name="DAM (hourly)", line=dict(color="#EF553B", width=1.5)))
fig.update_layout(
    title=f"DAM vs RTM settlement point price — {HUB}, June 2025",
    xaxis_title="", yaxis_title="$/MWh", height=420, legend=dict(orientation="h", y=1.08),
)
fig.show()

summary = con.execute("""
    SELECT 'DAM' AS market, count(*) AS n_intervals,
           round(avg(spp_usd_per_mwh), 2) AS avg_usd_mwh,
           round(min(spp_usd_per_mwh), 2) AS min_usd_mwh,
           round(max(spp_usd_per_mwh), 2) AS max_usd_mwh
    FROM silver_dam_spp WHERE location = ?
    UNION ALL
    SELECT 'RTM', count(*), round(avg(spp_usd_per_mwh), 2), round(min(spp_usd_per_mwh), 2),
           round(max(spp_usd_per_mwh), 2)
    FROM silver_rtm_spp WHERE location = ?
""", [HUB, HUB]).df()
summary

,market,n_intervals,avg_usd_mwh,min_usd_mwh,max_usd_mwh
0,DAM,720,34.62,13.26,182.12
1,RTM,2880,31.94,-0.14,1250.73


**Data quality is checked, not assumed.** Every silver table is validated for
duplicate timestamps, missing intervals, and correct interval counts across DST
transitions before anything downstream trusts it. Re-running the actual
`run_dq_checks` function from the pipeline (not a re-implementation) against each
table:

In [3]:
DQ_SPEC = {
    "dam_spp": ("1h", 24),
    "rtm_spp": ("15m", 96),
    "load": ("1h", 24),
    "dam_as_prices": ("1h", 24),
}

rows = []
for dataset, (freq, freq_per_day) in DQ_SPEC.items():
    df = con.execute(f"SELECT * FROM silver_{dataset}").pl()
    group_cols = [c for c in ("location", "product", "zone") if c in df.columns]
    tz = df.get_column("interval_start")[0].tzinfo
    start_dt = dt.datetime.combine(START, dt.time.min, tzinfo=tz)
    end_dt = dt.datetime.combine(END + dt.timedelta(days=1), dt.time.min, tzinfo=tz)
    report = run_dq_checks(df, dataset, "interval_start", start_dt, end_dt, freq, freq_per_day, group_cols)
    rows.append({
        "dataset": dataset,
        "duplicate_timestamps": len(report.duplicate_timestamps),
        "missing_intervals": len(report.missing_intervals),
        "dst_issues": len(report.dst_day_issues),
        "result": "CLEAN" if report.is_clean else "FAILED",
    })
pd.DataFrame(rows)

,dataset,duplicate_timestamps,missing_intervals,dst_issues,result
0,dam_spp,0,0,0,CLEAN
1,rtm_spp,0,0,0,CLEAN
2,load,0,0,0,CLEAN
3,dam_as_prices,0,0,0,CLEAN


## 2. The perfect-foresight benchmark

`solve_perfect_foresight()` is a linear program (CVXPY/HiGHS): given a realized price
series and the battery's physical/economic parameters (power, energy, round-trip
efficiency, degradation cost, daily cycle limit), it finds the charge/discharge
trajectory that maximizes revenue *with full hindsight*. That's the ceiling every
causal, no-lookahead strategy in M3 gets scored against.

Three variants, same battery, same month:
- **RTM-only** — energy arbitrage against 15-minute real-time prices
- **DAM-only** — energy arbitrage against hourly day-ahead prices
- **DAM + AS co-optimized** — day-ahead energy plus ancillary service capacity awards

The LP formulation, and why AS co-optimization is capacity-only (no deployment
energy, no SoC impact — a deliberate simplifying assumption), are documented in
[ADR 0006](../docs/adr/0006-perfect-foresight-lp-formulation.md) and
[ADR 0007](../docs/adr/0007-as-capacity-only-co-optimization.md).

In [4]:
rtm_ts, rtm_prices = load_spp_series(con, "silver_rtm_spp", HUB, START, END)
rtm_result = solve_perfect_foresight(rtm_ts, rtm_prices, battery, interval_hours=0.25)

dam_ts, dam_prices = load_spp_series(con, "silver_dam_spp", HUB, START, END)
dam_result = solve_perfect_foresight(dam_ts, dam_prices, battery, interval_hours=1.0)

as_prices = load_dam_as_price_series(con, dam_ts, START, END)
dam_as_result = solve_perfect_foresight(dam_ts, dam_prices, battery, interval_hours=1.0, as_prices_usd_per_mw=as_prices)

revenue = pd.DataFrame([
    {"variant": "RTM-only", "revenue_usd": rtm_result.total_revenue_usd},
    {"variant": "DAM-only", "revenue_usd": dam_result.total_revenue_usd},
    {"variant": "DAM + AS co-optimized", "revenue_usd": dam_as_result.total_revenue_usd},
])

fig = go.Figure(go.Bar(
    x=revenue["variant"], y=revenue["revenue_usd"],
    marker_color=["#636EFA", "#EF553B", "#00CC96"],
    text=revenue["revenue_usd"].map(lambda v: f"${v:,.0f}"), textposition="outside",
))
fig.update_layout(
    title=f"Perfect-foresight revenue by variant — {HUB}, June 2025, {battery.power_mw:.0f}MW/{battery.energy_mwh:.0f}MWh battery",
    yaxis_title="$", height=420, showlegend=False,
)
fig.show()
revenue

,variant,revenue_usd
0,RTM-only,371952.717743
1,DAM-only,244165.638212
2,DAM + AS co-optimized,517092.270809


**Read the DAM+AS number as an optimistic ceiling, not a realistic estimate** — a
battery that's never actually called to deliver AS is cheap for the LP to fully
commit, since there's no modeled downside. That's a documented, deliberate
simplification (ADR 0007), not an oversight — the kind of caveat this project is
built to surface explicitly rather than bury in a single headline number.

## 3. What the optimal dispatch actually looks like

Price, dispatch power (charge negative / discharge positive), and state of charge
for one week, RTM-only — the "buy low, sell high, respect the battery's physical
limits" pattern the LP found, made visible.

In [5]:
week_start, week_end = dt.datetime(2025, 6, 15), dt.datetime(2025, 6, 22)
idx = [i for i, ts in enumerate(rtm_ts) if week_start <= ts.replace(tzinfo=None) < week_end]

ts_week = [rtm_ts[i] for i in idx]
price_week = [rtm_prices[i] for i in idx]
net_power_week = [rtm_result.discharge_mw[i] - rtm_result.charge_mw[i] for i in idx]
soc_week = [rtm_result.soc_mwh[i] for i in idx]

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                     subplot_titles=("RTM price ($/MWh)", "Dispatch power (MW, discharge +)", "State of charge (MWh)"))
fig.add_trace(go.Scatter(x=ts_week, y=price_week, line=dict(color="#636EFA", width=1)), row=1, col=1)
fig.add_trace(go.Bar(x=ts_week, y=net_power_week, marker_color="#EF553B"), row=2, col=1)
fig.add_trace(go.Scatter(x=ts_week, y=soc_week[:-1], line=dict(color="#00CC96", width=1.5), fill="tozeroy"), row=3, col=1)
fig.update_layout(title="One week of perfect-foresight dispatch — RTM-only", height=650, showlegend=False)
fig.show()

## 4. Engineering practices behind the numbers

A few things a code review would find, beyond the notebook:

- **Typed configuration, not dicts** — `BatterySpec` and market configs are pydantic
  models with validated field constraints (e.g. `min_soc_fraction < max_soc_fraction`
  is enforced at construction, not discovered at runtime).
- **Golden-case tests derived by hand before running the solver** — six cases in
  `tests/test_perfect_foresight.py` (frictionless two-cycle arbitrage, efficiency
  losses, a binding daily cycle limit, an unprofitable-given-degradation case, AS
  capacity revenue, and an invariant that AS co-optimization never reduces the
  energy-only baseline), each with an expected dollar figure worked out
  independently of the code, so the tests can actually catch a wrong LP formulation
  rather than just mirroring it.
- **Data quality gates as a first-class pipeline step**, not an afterthought — see
  Section 1 above, run against the real function, not reimplemented for this
  notebook.
- **Architecture decision records** for every non-obvious tradeoff — why a pure LP
  and not a MILP, why AS is capacity-only, why HB_HOUSTON is the default hub — each
  with the cost of the decision written down, not just the choice.
- **`ruff` and `mypy --strict` clean**, one draft PR per milestone, each with its own
  write-up in `docs/milestones/`.

Running the actual test suite live:

In [6]:
import subprocess

result = subprocess.run(
    ["uv", "run", "pytest", "-q"], capture_output=True, text=True, cwd=Path.cwd(),
)
print(result.stdout[-800:])

....................................                                     [100%]
36 passed in 26.77s



## What's next

| # | Milestone | Status |
|---|---|---|
| M1 | Ingestion + data quality | Done |
| M2 | Perfect-foresight benchmark | Done |
| M3 | Causal strategies (DA-committed, rolling-horizon MPC) + walk-forward backtest engine, structurally no-lookahead | Planned |
| M4 | Full history across ERCOT's RTC+B market redesign boundary | Planned |
| M5 | Polish — HTML reports, one-command reproduction, `v0.1.0` | Planned |

M3 is where this project's actual thesis gets tested: can a strategy that only sees
data up to time *t* capture a meaningful fraction of the hindsight ceiling computed
above? That number — **% of perfect-foresight revenue captured** — is the whole
point of the project.

---
Repo: [`ercot-bess-lab`](..) · Milestone write-ups: [`docs/milestones/`](../docs/milestones/) ·
Decisions: [`docs/adr/`](../docs/adr/) · Per-milestone detail notebooks:
[M1](01_m1_data_showcase.ipynb), [M2](02_m2_perfect_foresight_showcase.ipynb)